In [ ]:
#빅카인즈 뉴스 정보 수집하기

#Step 1. 필요한 모듈을 로딩합니다
import time
import sys                 
import pandas  as pd    
import math 
import os
import random
import pyautogui 
import numpy  

from bs4 import BeautifulSoup
from selenium import webdriver
from selenium.webdriver.support.ui import Select
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC


#Step 2. 필요한 정보를 수집합니다
query_txt_1 = input('1.크롤링할 뉴스의 키워드는 무엇입니까?: ')
query_txt = query_txt_1.replace('"','')
poham_keyword = input('2.반드시 포함해야 할 단어가 있다면 적어주세요(여러 단어일 경우 , 로 구분해 주세요): ')
no_keyword = input('3.제외할 단어가 있다면 적어주세요:' )

start_date = input('4.조회를 시작할 날짜를 입력하세요(기본값:2026-03-01): ')
if start_date == '' :
    start_date = '2026-03-01'
    
end_date = input('5.조회를 종료할 날짜를 입력하세요(기본값:2026-03-10): ')
if end_date == '' :
    end_date = '2026-03-10'
    
f_dir = input("6.파일을 저장할 폴더명만 쓰세요(기본값:c:\\py_temp\\):")
if f_dir == '' :
    f_dir="c:\\py_temp\\"

print("\n")

print("요청하신 데이터를 수집 중입니다")
print("\n")
print("잠시만 기다려 주세요~~~~~^^")
print("\n")

#Step 3.저장될 파일위치와 이름을 지정합니다
now = time.localtime()
t = '%04d-%02d-%02d-%02d-%02d-%02d' % (now.tm_year, now.tm_mon, now.tm_mday, now.tm_hour, now.tm_min, now.tm_sec)

os.makedirs(f_dir+'Bigkinds'+'-'+t+'-'+query_txt)

fx_name = f_dir+'Bigkinds'+'-'+t+'-'+query_txt+'\\'+'Bigkinds'+'-'+query_txt+'.xlsx'
fc_name = f_dir+'Bigkinds'+'-'+t+'-'+query_txt+'\\'+'Bigkinds'+'-'+query_txt+'.csv'

In [23]:
# Step 4. 크롬 드라이버 설정 
s = Service("c:/py_temp/chromedriver.exe")
driver = webdriver.Chrome(service=s)

# Step 5. 빅카인즈 사이트에 접속합니다
driver.get('https://www.bigkinds.or.kr/')
driver.maximize_window( )
print("Step 1.빅카인즈 홈페이지에 자동 접속 합니다. 잠시만 기다려 주세요 ^^")
time.sleep(random.randrange(5,10))
   
# 팝업창이 있을 경우 닫기
from selenium.webdriver.common.by import By

buttons = driver.find_elements(By.CSS_SELECTOR, 'button[data-name="popup-dismiss"]')

for btn in buttons:
    try:
        btn.click()
    except:
        pass
        
print('Step 2.날짜 조건과 검색어를 입력하여 요청하신 뉴스를 검색중입니다. 잠시 후 총 뉴스의 건수가 표시될 예정입니다~^^')

#Step 6. 검색 조건 입력하기
# 검색어 입력한 후 검색하기    
element = driver.find_element(By.ID,"total-search-key")
driver.find_element(By.ID,"total-search-key").click( )
element.send_keys(query_txt)
#element.send_keys("\n")
time.sleep(1)

# 상세검색 버튼 클릭
driver.find_element(By.ID,'ig-sd-btn').click()

# 기간버튼 클릭하기
driver.find_element(By.XPATH,'//*[@id="ds-modal"]/div[3]/div/div[2]/ul/li[2]/a').click()

# 시작 날짜 입력하기
import pyautogui as pg
from selenium.webdriver.common.keys import Keys

s_date = driver.find_element(By.ID,'search-begin-date') 
driver.find_element(By.ID,'search-begin-date').click()

s_date.send_keys(Keys.CONTROL, 'a') 
s_date.send_keys(Keys.BACKSPACE)
s_date.send_keys(start_date)
time.sleep(1)

# 종료 날짜 입력하기    
e_date = driver.find_element(By.ID,'search-end-date') 
driver.find_element(By.ID,'search-end-date').click()

e_date.send_keys(Keys.CONTROL, 'a') 
e_date.send_keys(Keys.BACKSPACE)
e_date.send_keys(end_date)
e_date.send_keys('\n')
time.sleep(1)

# 상세 검색 버튼 누르기
driver.find_element(By.XPATH,'//*[@id="ds-modal"]/div[3]/div/div[2]/ul/li[6]/a').click()

# 단어중 1개이상 포함하는 단어 입력
orkeyword = driver.find_element(By.ID,'orKeyword1')
orkeyword.click()
orkeyword.clear()
orkeyword.send_keys(poham_keyword)
time.sleep(1)

# 제외할 단어
notkeyword = driver.find_element(By.ID,'notKeyword1')  
notkeyword.click()
notkeyword.clear()
notkeyword.send_keys(no_keyword)
time.sleep(1)

# 아래의 적용하기 버튼 클릭하여 검색 하기
driver.find_element(By.XPATH,'//*[@id="ds-modal"]/div[3]/div/div[8]/div[2]/div/button[2]').click()
time.sleep(10)

# 상단의 최신순/과거순 선택하기
select = Select(driver.find_element(By.ID,'select1'))
select.select_by_visible_text("과거순")   # 과거순
time.sleep(2)

# 한꺼번에 출력될 건수 선택하기(10건 -> 100건)
select2 = Select(driver.find_element(By.ID,'select2'))
select2.select_by_value("100")   #100건씩 출력
time.sleep(10)

# Step 7. 크롤링 건수 추출 후 건수 입력받기
time.sleep(0.2)
total = driver.find_element(By.XPATH,'//*[@id="news-results-tab"]/div[2]/h3/span[6]').text
print("Step 3.검색된 기사의 개수는 총 %s 건 입니다" %total)
cnt = int(input('검색된 기사 중에서 크롤링 할 건수를 입력해주세요(최대:%s 건): ' %total) )
page_cnt = math.ceil(cnt / 100) 
print("\n")

# step 8. 본문 내용 추출하기
company2=[ ]     # 언론사명 컬럼
title2=[ ]       # 게시글 제목 컬럼
date2=[ ]        # 게시글 날짜 컬럼
content2=[ ]     # 기사 본문 컬럼
url2=[]          # 기사 URL 추출하기

no = 1           # 전체 게시글 번호

# 마우스 휠을 자동으로 돌리는 사용자정의함수
def scroll_down(driver):      
    driver.execute_script("window.scrollBy(0,400);")
    time.sleep(1)

def scroll_up(driver):      
    driver.execute_script("window.scrollBy(0,-1500);")
    time.sleep(1)

# 화면 맨 아래로 내리는 함수
def scroll_down2(driver):
  driver.execute_script("window.scrollTo(0,document.body.scrollHeight);")
  time.sleep(3)
        
# 페이지 번호 변경하는 for 반복문
for a in range(1,page_cnt+1) :
    news_title = 1   # 뉴스 제목 순서용 번호
    
    # 한 페이지에서 뉴스 글을 변경하는 for 반복문
    for b in range(0,100) :
        
        print('%s 번째 기사 정보를 수집합니다=========================================================' %no)
        
        html = driver.page_source
        soup = BeautifulSoup(html, 'html.parser')
        news_list = soup.find('div','news-wp type-wz no-rev').find_all('div','news-item')
        
        # 1. 언론사 정보 추출
        try :
            company = news_list[b].find('div','info').find('a').get_text().split('\n')
            company1 = company[1].strip()
        except :
            company = news_list[b].find('div','info').get_text().split('\n')  
            company1 = company[2].strip()
        
        company2.append(company1)
        print('1.언론사명: ' , company1)      
        
        # 2. 기사 제목 추출
        title = news_list[b].find('span','title-elipsis').get_text().replace('\n','').strip()
        title2.append(title)
        print('2.뉴스제목:' , title)
        
        # 3. 뉴스 보도 날짜
        date1 = news_list[b].find('p','name')
        date = date1.get_text().strip().replace('-','').replace('>','')

        if date == '' or len(date) != 10:
                date = date1[3].get_text().strip().replace("\n","")

        date2.append(date)
        print("3.게시날짜 : %s" %date)
        
        # 4. 뉴스 본문
        driver.find_element(By.XPATH,'//*[@id="news-results"]/div[%s]/div/div[2]/a/div/strong/span' %news_title).click()
        news_title += 1
        time.sleep(2)
        
        html2 = driver.page_source
        soup2 = BeautifulSoup(html2, 'html.parser')
        
        cont=soup2.find("div","news-view-body").get_text().replace("\n","").strip()
        if len(cont) == 0 :
            cont = '내용이 없는 기사입니다'

        content2.append(cont)
        print("4.뉴스내용 : %s" %cont)

        scroll_down(driver)
        time.sleep(2)
                  
        # 5. 원문 URL 추출하기
        try:
            url1 = news_list[b].find('div','info').find('a')['href']
        except :
            url1 = ' URL 이 없습니다'
            
        url2.append(url1)
        print("5.기사 URL : %s" %url1)

        # 본문 기사 하단의 닫기 버튼 클릭##
        from selenium.webdriver.common.by import By
        import time
        
        clicked = False
        
        for i in range(10):  # 최대 10번 아래로 스크롤
            buttons = driver.find_elements(
                By.XPATH,
                "//button[@data-dismiss='modal' and contains(normalize-space(.), '닫기')]"
            )
        
            for btn in buttons:
                try:
                    if btn.is_displayed() and btn.is_enabled():
                        driver.execute_script("arguments[0].scrollIntoView({block: 'center'});", btn)
                        time.sleep(0.5)
                        driver.execute_script("arguments[0].click();", btn)
                        clicked = True
                        break
                except:
                    pass
        
            if clicked:
                break
        
            driver.execute_script("window.scrollBy(0, 500);")
            time.sleep(1)
        
        if not clicked:
            print("닫기 버튼을 찾지 못했거나 클릭하지 못했습니다.")
        ####

        if no == cnt :
            break
        no += 1
        
        print('\n')

        scroll_up(driver)
        
    time.sleep(random.randrange(2,6))
    
    a += 1 
   
    scroll_down2(driver)
    
    # time.sleep(2)
    # driver.find_element(By.XPATH,'//*[@id="news-results-tab"]/div[6]/div[2]/div/div/div/div/div[4]/').click()


    time.sleep(10)
            

# Step 9. 출력 결과 저장하기
newsall = pd.DataFrame()
newsall['언론사']=pd.Series(company2)
newsall['뉴스제목']=pd.Series(title2)
newsall['게시날짜']=pd.Series(date2)
newsall['뉴스내용']=pd.Series(content2)
newsall['뉴스URL']=pd.Series(url2)

#저장하기
newsall.to_excel(fx_name,index=False)
newsall.to_csv(fc_name,encoding="utf-8-sig",index=False)
print('=' *60)
print('1.xlsx 파일저장경로: %s ' %fx_name)
print('2. csv 파일저장경로: %s ' %fc_name)

Step 1.빅카인즈 홈페이지에 자동 접속 합니다. 잠시만 기다려 주세요 ^^
Step 2.날짜 조건과 검색어를 입력하여 요청하신 뉴스를 검색중입니다. 잠시 후 총 뉴스의 건수가 표시될 예정입니다~^^
Step 3.검색된 기사의 개수는 총 11,348 건 입니다


검색된 기사 중에서 크롤링 할 건수를 입력해주세요(최대:11,348 건):  3




1 번째 기사 정보를 수집합니다=========================================================
1.언론사명:  브릿지경제
2.뉴스제목: 이스라엘, 이란 테헤란 폭격⋯ 하메네이의 집무실 인근
3.게시날짜 : 2026/03/01
4.뉴스내용 : 공습을 받아 연기에 둘러싸인 이란의 모습. 연합뉴스이스라엘이 28일(현지시간) 이란을 상대로 전격적인 군사 공격을 감행하면서 중동 정세가 급격히 악화되고 있다.     미국과 이란 간 군사력 억제 협상이 난항을 겪는 가운데 발생한 이번 공습으로 양국 간 충돌 우려가 다시 커지고 있다.     이날 AP통신과 로이터통신 등에 따르면 이스라엘 국방부 장관은 이날 이란을 대상으로 예방적 미사일 공격을 실시했다고 발표했다. 이스라엘 군은 이란의 보복 공격 가능성에 대비해 국민들에게 경계를 당부하며 본토 전역에 방공 사이렌을 울렸다.     이스라엘 정부는 공습 직후 자국 내 사업장 폐쇄와 휴교령도 함께 발표하며 비상 대응 체제에 들어갔다. 이란 수도 테헤란에서는 대규모 폭발과 함께 짙은 연기가 치솟았다고 이란 국영 TV가 보도했다.     외신들은 폭발 지점이 이란 최고지도자 아야톨라 세예드 알리 하메네이의 집무실 인근이라고 봤지만, 당시 하메네이의 위치나 안전 여부는 확인되지 않았다.     최근 미국과 긴장이 고조된 이후 하메네이는 공식 석상에 모습을 드러내지 않은 상태였다. 이스라엘이 언급한 예방타격은 상대를 먼저 공격한다는 점에서는 선제타격과 유사하지만, 위협이 임박하기 전에 잠재적 위험을 제거한다는 군사 전략이라는 점에서 차이가 있다.     이스라엘은 그동안 이란이 드론과 미사일 전력을 강화하고 핵무기 개발을 추진하면서 자국 안보를 위협하고 있다고 주장해왔다. 이번 공격은 핵 프로그램과 탄도미사일 문제를 둘러싸고 미국과 이란 간 긴장이 최고조에 이른 상황에서 발생했다.     도널드 트럼프 미국 대통령은 외교적 해법을 우선하면서도 군사행동 가능성을 배제하지 않는 입장을 유지해왔